# 과제 — 보안 공지 요약 봇

긴 영문 보안 공지(advisory)를 넣으면 **네 줄**(제품·영향·조치·긴급도)로 요약하고,
**어느 모델이 가장 잘하는지 숫자로** 비교한다.

**비교 모델 4개** — 클라우드 2개(`gpt-4.1-mini`, `gpt-4.1-nano`) + 로컬 2개(`qwen2.5:3b`, `llama3.2`)

**전체 흐름**

```
공지문 → 모델별 요약 → 채점(형식·제품·조치) + 토큰·시간·잘림 → 결과 저장 → 결론
```

---

## 1. 환경 설정 — 라이브러리 · API 키 · 클라이언트

- `.env` 에서 `OPENAI_API_KEY`, `OLLAMA_URL` 을 읽는다.
- 로컬(Ollama) 모델도 **같은 `OpenAI` 클래스**로 부른다 — `base_url` 만 다르다.

In [31]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI

In [32]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


In [33]:
openai = OpenAI()

In [34]:
ollama_url = os.getenv('OLLAMA_URL')

In [35]:
qwen = OpenAI(api_key="ollama", base_url=ollama_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

## 2. 모델을 데이터로 다루기

모델 4개를 **딕셔너리 한 곳**에 모은다. `{이름: (클라이언트, 모델명)}` 구조라
모델을 추가·제거할 때 **이 딕셔너리에 한 줄만** 고치면 된다.
호출 함수는 클라우드인지 로컬인지 **몰라도 되게** 만든다.

In [36]:
models = {
    "gpt-4.1-mini": (openai, "gpt-4.1-mini"),
    "gpt-4.1-nano": (openai, "gpt-4.1-nano"),
    "qwen2.5:3b":   (ollama, "qwen2.5:3b"),
    "llama3.2":     (ollama, "llama3.2"),
}

## 3. 데이터 불러오기

`advisories_sample.json` 은 실제 공개 공지 10건이다.
`pd.read_json` 으로 표(DataFrame)로 읽는다. 모델에게는 **`advisory`(본문)만** 준다.

> 채점용 정답인 `label`(제품명)은 모델에게 **주지 않는다.** 제품명도 모델이 본문에서 스스로 찾아야 한다.

In [37]:
import pandas as pd

df = pd.read_json('advisories_sample.json') # 표로 읽힘

advisory_text = df['advisory'] # 공지 컬럼만

## 4. 요약 프롬프트 (system)

출력 형식을 **정확히 네 줄**로 못박고, 두 가지 안전장치를 넣는다.

- **"본문에 없는 내용은 지어내지 마라"** — 환각 방지
- **버전이 없으면 `공지에 없음`** 이라고 쓰게 미리 지정 — 4·8번(패치 버전 없는 공지) 대비

이 두 줄이 있고 없고가 **지어내기 관찰**(아래 11번)의 핵심이다.

In [38]:
SYSTEM_PROMPT = """너는 보안 공지(security advisory)를 요약하는 분석가다.
주어진 공지 본문만 근거로 삼는다. 본문에 없는 내용은 절대 지어내지 않는다.
너의 사전 지식으로 버전 번호나 제품 정보를 보충하지 마라.

반드시 아래 네 줄 형식으로만 답한다. 다른 문장, 인사말, 설명, 코드블록을 붙이지 마라.

제품: <제품 이름과 영향받는 버전>
영향: <공격자가 무엇을 할 수 있나, 한 문장>
조치: <올려야 할 버전. 본문에 패치 버전이 없으면 반드시 "공지에 없음">
긴급도: <긴급|높음|보통|낮음 중 하나>

규칙:
- '조치'에 들어갈 버전이 본문에 명시되어 있지 않으면 추측하지 말고 정확히 "공지에 없음"이라고 쓴다.
- '긴급도'는 네 단어 중 하나만 쓴다.
- 모든 답은 한국어로 쓴다."""


## 5. 호출 함수 (1차) · 스모크 테스트

먼저 **답변 텍스트만** 돌려주는 간단한 버전으로 1건만 돌려 본다.
`temperature=0`(같은 공지→같은 답), `max_tokens=300`(네 줄엔 충분), `try/except`(한 모델이 죽어도 나머지는 계속).

> 전체 10건×4모델(=40회)을 돌리기 전에, **1건으로 형식이 제대로 나오는지** 눈으로 먼저 확인한다.

In [39]:
def summarize_one(advisory_text):
    """공지 1건을 받아 모델별 요약 dict 반환"""
    user_prompt = f"다음 보안 공지를 네 줄로 요약하라.\n\n---\n{advisory_text}\n---"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_prompt},
    ]
    responses = {}
    for name, (client, model) in models.items():
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=0,       # ← 필수
                max_tokens=300,      # ← 잘림 검사용
            )
            responses[name] = resp.choices[0].message.content
        except Exception as e:
            responses[name] = f"[Error] {e}"
    return responses


In [40]:
test = summarize_one(df['advisory'][0])   # 1번 공지 (Log4j)만

for name, content in test.items():
    print(f"===== {name} =====")
    print(content)
    print()

===== gpt-4.1-mini =====
제품: Apache Log4j 2.x (2.14.1 이하 및 2.15.0 이전 버전)
영향: 공격자가 LDAP JNDI 파서를 통해 원격 코드 실행을 할 수 있다.
조치: 2.16.0 이상으로 업데이트
긴급도: 긴급

===== gpt-4.1-nano =====
제품: Log4j 버전 2.14.1 이전
영향: 원격 코드 실행(RCE) 공격이 가능
조치: 공지에 없음
긴급도: 높음

===== qwen2.5:3b =====
제품: Apache Log4j 2.0
영향: 로그 메시지나 파라미터에서 사용자 또는 불안정한 데이터를 로깅할 경우 로그4j의 특정 문제로 인해 원격 코드 실행(RCE)이 가능합니다.
조치: 공지에 없음
긴급도: 높음

===== llama3.2 =====
# 제품: Log4j
#영향: attacker-controlled LDAP와 JNDI 관련 엔드포인트를 사용하여 원격 코드 실행을 할 수 있습니다.
#조치: Log4j 2.16.0
#긴급도: 높음



## 6. 호출 함수 (최종) — 측정값까지 담기

형식이 OK인 걸 확인했으니, 이제 **채점·비교에 필요한 숫자**까지 함께 기록하도록 함수를 키운다.

| 담는 값 | 쓰는 곳 |
|---|---|
| `content` | 채점 (형식·제품·조치) |
| `finish_reason` | 잘림 검사 (`length` 면 답이 잘린 것) |
| `in_tokens` / `out_tokens` | 토큰 집계 |
| `seconds` | 로컬 모델의 '비용'은 돈이 아니라 **시간** |

In [41]:
import time

def summarize_one(advisory_text):
    user_prompt = f"다음 보안 공지를 네 줄로 요약하라.\n\n---\n{advisory_text}\n---"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_prompt},
    ]
    responses = {}
    for name, (client, model) in models.items():
        try:
            start = time.time()                      # ← 시간 측정 시작
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=0,
                max_tokens=300,
            )
            responses[name] = {
                "content":       resp.choices[0].message.content,
                "finish_reason": resp.choices[0].finish_reason,   # ← 잘림 검사 (필수 4)
                "in_tokens":     resp.usage.prompt_tokens,        # ← 토큰 (필수 3)
                "out_tokens":    resp.usage.completion_tokens,
                "seconds":       round(time.time() - start, 2),   # ← 로컬=비용
            }
        except Exception as e:
            responses[name] = {"content": None, "error": str(e)}
    return responses


## 7. 10건 전체 요약 실행

공지 10건을 모두 돌려 결과를 `all_results` 에 모은다.
구조는 `{cve_id: {모델명: {content, finish_reason, in_tokens, ...}}}`.

> 로컬 모델이 느려 수십 초~수 분 걸릴 수 있다. 한 번 돌려 두면 아래 분석 셀들은 **다시 호출하지 않고** 이 결과만 쓴다.

In [42]:
all_results = {}   # {cve_id: {모델명: {content, finish_reason, ...}}}

for i in range(len(df)):
    cve = df['cve_id'][i]
    print(f"처리 중... [{i+1}/{len(df)}] {cve}")
    all_results[cve] = summarize_one(df['advisory'][i])

print("완료!")


처리 중... [1/10] CVE-2021-44228
처리 중... [2/10] CVE-2022-22965
처리 중... [3/10] CVE-2017-5638
처리 중... [4/10] CVE-2014-6271
처리 중... [5/10] CVE-2023-49105
처리 중... [6/10] CVE-2026-48710
처리 중... [7/10] CVE-2026-59822
처리 중... [8/10] CVE-2026-75650
처리 중... [9/10] CVE-2026-49869
처리 중... [10/10] CVE-2026-73570
완료!


## 8. 채점 — 형식 · 제품 · 조치

`KEY` 는 과제가 준 정답표다. 세 가지를 **낱말 포함**으로 판정한다.

| 항목 | 판정 |
|---|---|
| **형식** | `제품:` `영향:` `조치:` `긴급도:` 네 라벨이 모두 있는가 |
| **제품** | 정답 제품 낱말이 답에 들어 있는가 |
| **조치** | 정답 버전 낱말이 들어 있는가 (4·8번은 `공지에 없음` 이어야 정답) |

`score_df` 는 모델별 **/10** 점수표가 된다.

In [43]:
KEY = {
    "CVE-2021-44228": {"product": "Log4j",          "fix": ["2.16.0"]},
    "CVE-2022-22965": {"product": "Spring",         "fix": ["5.3.18", "5.2.20"]},
    "CVE-2017-5638":  {"product": "Struts",         "fix": ["2.3.32", "2.5.10.1"]},
    "CVE-2014-6271":  {"product": "Bash",           "fix": None},   # 없음이 정답
    "CVE-2023-49105": {"product": "ownCloud",       "fix": ["10.13.1"]},
    "CVE-2026-48710": {"product": "Starlette",      "fix": ["1.0.1"]},
    "CVE-2026-59822": {"product": "LiteLLM",        "fix": ["1.84.0"]},
    "CVE-2026-75650": {"product": "Adobe Commerce", "fix": None},   # 없음이 정답
    "CVE-2026-49869": {"product": "Kestra",         "fix": ["1.0.45", "1.3.21"]},
    "CVE-2026-73570": {"product": "Zimbra",         "fix": ["10.1.20"]},
}

def grade(cve, content):
    if not content:
        return {"형식": 0, "제품": 0, "조치": 0}
    low = content.lower()
    fmt  = all(lbl in content for lbl in ["제품:", "영향:", "조치:", "긴급도:"])
    prod = KEY[cve]["product"].lower() in low
    fix  = KEY[cve]["fix"]
    if fix is None:
        act = "공지에 없음" in content        # 4·8번은 '없음'이 정답
    else:
        act = any(v.lower() in low for v in fix)
    return {"형식": int(fmt), "제품": int(prod), "조치": int(act)}


In [44]:
rows = []
for name in models:
    f = p = a = err = 0
    for cve, per_model in all_results.items():
        item = per_model[name]
        if item.get("content") is None:   # 호출 실패
            err += 1
            continue
        g = grade(cve, item["content"])
        f += g["형식"]; p += g["제품"]; a += g["조치"]
    rows.append({"모델": name, "형식": f, "제품": p, "조치": a, "오류": err})

score_df = pd.DataFrame(rows)
score_df


,모델,형식,제품,조치,오류
0,gpt-4.1-mini,10,10,10,0
1,gpt-4.1-nano,10,10,7,0
2,qwen2.5:3b,10,10,8,0
3,llama3.2,10,10,7,0


## 9. 토큰 세기 — tiktoken (보내기 전)

`tiktoken` 으로 **호출하기 전에** 입력 토큰을 미리 센다. (응답의 `usage` 는 호출 후 실제값)

> ⚠️ `tiktoken` 은 인코딩 파일을 인터넷에서 받는다. 망이 막히면 타임아웃 나므로,
> 저장소 루트의 `tiktoken_cache`(미리 받아둔 `o200k_base`)를 쓰도록 `TIKTOKEN_CACHE_DIR` 을 먼저 지정한다.
> `o200k_base` 는 gpt-4.1 계열이 실제로 쓰는 인코딩이라 토큰 수도 더 정확하다.

In [46]:
# 필수 3 — tiktoken 으로 '보내기 전' 입력 토큰 미리 세기
# tiktoken 은 인코딩 파일을 인터넷에서 받는데, 망이 막히면 타임아웃 난다.
# 저장소 루트의 tiktoken_cache(미리 받아둔 파일)를 쓰도록 환경변수를 먼저 지정한다.
import os
os.environ["TIKTOKEN_CACHE_DIR"] = os.path.abspath("../../tiktoken_cache")
print("TIKTOKEN_CACHE_DIR =", os.environ["TIKTOKEN_CACHE_DIR"])

import tiktoken
enc = tiktoken.get_encoding("o200k_base")   # gpt-4.1 계열 인코딩 (캐시에 있는 파일)

print("\n공지별 입력 토큰 (보내기 전 예측):")
for i in range(len(df)):
    n = len(enc.encode(df['advisory'][i]))
    print(f"  {df['cve_id'][i]:18s} {n:5d} tokens")

total_input = sum(len(enc.encode(df['advisory'][i])) for i in range(len(df)))
print(f"\n전체 공지 입력 토큰 합계: {total_input}")

TIKTOKEN_CACHE_DIR = c:\Users\o3o_1\Desktop\Wins\lesson\ai\wins-ai\tiktoken_cache

공지별 입력 토큰 (보내기 전 예측):
  CVE-2021-44228       600 tokens
  CVE-2022-22965       444 tokens
  CVE-2017-5638        384 tokens
  CVE-2014-6271        181 tokens
  CVE-2023-49105        85 tokens
  CVE-2026-48710       483 tokens
  CVE-2026-59822       371 tokens
  CVE-2026-75650        61 tokens
  CVE-2026-49869       161 tokens
  CVE-2026-73570        80 tokens

전체 공지 입력 토큰 합계: 2850


## 10. 토큰 · 시간 · 잘림 집계

`all_results` 를 모델별로 합쳐 `usage_df` 를 만든다.

- **총토큰** = 입력 + 출력 (비용·부하의 지표)
- **시간(초)** = 로컬 모델에서는 이게 곧 비용
- **잘림** = `finish_reason == "length"` 인 건수 → **0 이어야** 공정한 비교. 많으면 `max_tokens` 를 올려 다시 돌린다.

In [47]:
# 필수 3·4 — 모델별 토큰 · 시간 · 잘림(length) 집계
usage_rows = []
for name in models:
    in_tok = out_tok = truncated = err = 0
    secs = 0.0
    for cve, per_model in all_results.items():
        item = per_model[name]
        if item.get("content") is None:      # 호출 실패
            err += 1
            continue
        in_tok  += item["in_tokens"]
        out_tok += item["out_tokens"]
        secs    += item["seconds"]
        if item["finish_reason"] == "length":   # 답이 잘림
            truncated += 1
    usage_rows.append({
        "모델": name,
        "입력토큰": in_tok,
        "출력토큰": out_tok,
        "총토큰": in_tok + out_tok,
        "시간(초)": round(secs, 1),
        "잘림": truncated,
        "오류": err,
    })

usage_df = pd.DataFrame(usage_rows)
usage_df

,모델,입력토큰,출력토큰,총토큰,시간(초),잘림,오류
0,gpt-4.1-mini,5521,622,6143,10.7,0,0
1,gpt-4.1-nano,5521,537,6058,8.3,0,0
2,qwen2.5:3b,5849,790,6639,82.2,0,0
3,llama3.2,5726,611,6337,68.8,0,0


## 11. 지어내기 관찰 — 4번 · 8번 (가장 중요)

10건 중 **두 건(4번 Bash · 8번 Adobe Commerce)은 공지에 패치 버전이 없다.**
정답은 `조치: 공지에 없음`.

모델이 그럴듯한 버전을 **지어내면** — 틀린 것보다 나쁘다. 사용자가 걸러낼 수 없기 때문이다.
각 모델이 이 두 건에서 `공지에 없음` 이라고 했는지, 아니면 없는 버전을 만들어냈는지 확인한다.

In [48]:
# 3장 — 패치 버전이 '없는' 4·8번에서 모델이 버전을 지어냈는가 (20점)
print("정답이 '조치: 공지에 없음' 인 두 건\n")
for cve in ["CVE-2014-6271", "CVE-2026-75650"]:   # 4번 Bash, 8번 Adobe Commerce
    print(f"######## {cve} ########")
    for name in models:
        content = all_results[cve][name].get("content") or ""
        made_up = "공지에 없음" not in content
        mark = "지어냄 의심 ⚠️" if made_up else "없음 ✅"
        action = next((ln.strip() for ln in content.splitlines() if "조치" in ln), "(조치 줄 없음)")
        print(f"  [{name:12s}] {mark}  |  {action}")
    print()

정답이 '조치: 공지에 없음' 인 두 건

######## CVE-2014-6271 ########
  [gpt-4.1-mini] 없음 ✅  |  조치: 공지에 없음
  [gpt-4.1-nano] 없음 ✅  |  조치: 공지에 없음
  [qwen2.5:3b  ] 없음 ✅  |  조치: 공지에 없음
  [llama3.2    ] 지어냄 의심 ⚠️  |  조치: GNU Bash 4.4

######## CVE-2026-75650 ########
  [gpt-4.1-mini] 없음 ✅  |  조치: 공지에 없음
  [gpt-4.1-nano] 없음 ✅  |  조치: 공지에 없음
  [qwen2.5:3b  ] 없음 ✅  |  조치: 공지에 없음
  [llama3.2    ] 없음 ✅  |  조치: 공지에 없음



## 12. 결과 저장 — `exercise_result.md`

점수표 · 토큰표 · 모델별 전체 답변을 `exercise_result.md` 로 저장한다.

- 맨 위에 **실행 시각**을 적는다 (모델은 계속 바뀌므로)
- 파일 저장 시 **`encoding="utf-8"`** 필수 (한글 깨짐 방지)

In [49]:
# 필수 6 — 결과를 exercise_result.md 로 저장
from datetime import datetime

def df_to_md(d):   # tabulate 없이 DataFrame → 마크다운 표
    cols = list(d.columns)
    out = ["| " + " | ".join(cols) + " |",
           "| " + " | ".join(["---"] * len(cols)) + " |"]
    for _, row in d.iterrows():
        out.append("| " + " | ".join(str(row[c]) for c in cols) + " |")
    return "\n".join(out)

with open("exercise_result.md", "w", encoding="utf-8") as f:   # encoding 필수
    f.write("# 보안 공지 요약 봇 — 실행 결과\n\n")
    f.write(f"실행 시각: {datetime.now():%Y-%m-%d %H:%M:%S}\n\n")

    f.write("## 모델별 정확도 (각 10점 만점)\n\n")
    f.write(df_to_md(score_df) + "\n\n")

    f.write("## 모델별 토큰 · 시간 · 잘림\n\n")
    f.write(df_to_md(usage_df) + "\n\n")

    f.write("## 모델별 전체 답변\n\n")
    for cve, per_model in all_results.items():
        f.write(f"### {cve}\n\n")
        for name in models:
            content = per_model[name].get("content") or "(응답 없음)"
            f.write(f"**{name}**\n\n```\n{content}\n```\n\n")

print("exercise_result.md 저장 완료!")

exercise_result.md 저장 완료!


## 결론

> 아래는 **직접 채우세요** — `score_df`, `usage_df`, 지어내기 검사 결과를 보고 판단합니다.

1. **어느 모델을 고르겠는가, 근거는?**
   - (예: 조치 점수 10/10, 지어냄 0건, 총토큰 6143, 시간 10.7초 → **gpt-4.1-mini** 선택)

2. **4번·8번에서 지어낸 모델이 있었는가?**
   - (지어내기 검사 셀 결과: llama3.2가 4번(CVE-2014-6271)에서 조치: <code>GNU Bash 4.4</code>로 없는 버전을 지어냈다.)

3. **점수가 같다면 무엇으로 고르겠는가? (토큰·속도)**
   - (gpt-4.1-nano)

4. **공지를 외부로 보낼 수 없는 조직이라면?**
   - 로컬 모델(`qwen2.5:3b` / `llama3.2`) 중 **Qwen** 를 쓴다.
   - 감수할 손해(숫자): 클라우드 대비 조치 점수 **2점** 낮고, 건당 **7.1초** 더 걸린다.